# Final Assignment - Part 2

### Assignment Instructions

#### Part 2   |   Advanced Difficulty Level

In this second part, you will be working with a datasets containing information on indian startups' funding, including the startup's details, the funding it received, and the investors' information.

In the usual `data` folder, you will find the following three datasets, including data from 2019, 2020 and 2021:

- `startup_funding2019.csv`
- `startup_funding2020.csv`
- `startup_funding2021.csv`

At [this link](https://www.kaggle.com/datasets/omkargowda/indian-startups-funding-data?select=startup_funding2019.csv) you can find the source of the original data (Kaggle) as well as more information on its composition *(note: the files in the data folder are slightly different from the originals)*.

1. Using a **for loop**, load all three .csv files in a temporary DataFrame called `df_tmp` and, at each cycle, add a new column named `Year` that includes the year of that csv file to the temporary table and append it to a final DataFrame named `fnd`. Your final `fnd` DataFrame should include the contents from all three csv files stacked one on top of the other.

    What is the **shape** of the `fnd` DataFrame?

In [18]:
import pandas as pd
import os

df_temp = []

for file in os.listdir('data/')[0:]:
    df = pd.read_csv(f'data/{file}')
    df['Year'] = file[-8:-4]
    df_temp.append(df)

fnd = pd.concat(df_temp)
print(f"The shape of the Database is {fnd.shape}")
#

The shape of the Database is (25113, 18)


2. If you check the columns' data types, you'll notice that the columns `Founded`, `Amount($)` and `Year` are being interpreted as strings instead of numbers. Format those three columns to numeric data types.

    What is the **total** `Amount($)` of funding given in the three years available?

In [3]:
colums_modified = ["Year", "Founded", "Amount($)"]

#Transform from str to float64
fnd[colums_modified] = fnd[colums_modified].apply(pd.to_numeric, errors="coerce")
#Fill NaN spots with the next valid observation
fnd[colums_modified] = fnd[colums_modified].bfill()

print(fnd['Amount($)'].sum())

24001000000.0


3. The following code shows us that "Inflection Point Ventures" was the `Investor` that funded the highest number of `Company/Brand`s overall (36 companies funded from 2019 to 2021).

    How did "Inflection Point Ventures" **rank** *(in terms of most `Company/Brand`s funded) **in 2020**? (Note: in the answer write the rank number, where 1 = most funded company)*

In [4]:
# run this cell (don not delete it)
fnd.groupby('Investor', as_index=False).size().sort_values('size', ascending=False).head(1)

,Investor,size
399,Inflection Point Ventures,25


In [10]:
investor_rank = fnd[fnd['Year'] == 2020].groupby('Investor', as_index=False).size().sort_values('size', ascending=False)
investor_rank['rank'] = investor_rank['size'].rank(method='max', ascending=False)
print(investor_rank[investor_rank['Investor'] == 'Inflection Point Ventures'])

Empty DataFrame
Columns: [Rating, Company Name, Job Title, Salary, Salaries Reported, Location, Employment Status, Job Roles, Year, Company/Brand, Founded, HeadQuarter, Sector, What it does, Founders, Investor, Amount($), Stage]
Index: []
Empty DataFrame
Columns: [Rating, Company Name, Job Title, Salary, Salaries Reported, Location, Employment Status, Job Roles, Year, Company/Brand, Founded, HeadQuarter, Sector, What it does, Founders, Investor, Amount($), Stage]
Index: []


4. Load the `Software Professionals Salary.csv` file in a DataFrame named `sps` (just like you did in Part 1), then perform the following tasks **and answer the question at the end**:
    1. starting from the `sps` DataFrame, create a new DF called `sps_loc` where you group by `Location` and show, for each city in the dataset, the average `Rating` and `Salary`;
    2. starting from the `fnd` DataFrame, create a new DF called `fnd_loc` where you group by `HeadQuarter` and show, for each city in the dataset **for the year 2021**, the total number of `Company/Brand`s funded and the total `Amount($)` invested;
    3. merge the two DataFrames you just created so to **keep just the cities that are in both datasets** and save the results in a third DataFrame called `sps_fnd_loc` *(note: make sure to use the correct type of join)*;
    4. using the `sps_fnd_loc` DataFrame:
        1. delete the `HeadQuarter` column
        2. create a new column `Amount($MM)` that is equal to `Amount($)` divided by 1,000,000
        3. delete the `Amount($)` column
        4. rename all the columns to the following names: `['City', 'Avg. Rating', 'Avg. Salary', 'Nr. Companies Funded', 'Sum Funding ($MM)']`
    
    **Question**: Look at the `City` that received the **highest** `Avg. Rating` score by employees: what is the `Nr. Companies Funded` in that city?

In [ ]:
locality = sps_loc.groupby(by='Location')[['Rating', 'Salary']].agg('mean')
fnd_loc = fnd[fnd['Year'] == 2021].copy()
investment = fnd_loc.groupby(by='HeadQuarter')[['Company/Brand', 'Amount($)']].agg({'Company/Brand' : 'count', 'Amount($)': 'sum'})

sps_fnd_loc = locality.reset_index().join(investment.reset_index(), how='inner')
sps_fnd_loc['Amount($MM)'] = sps_fnd_loc['Amount($)'] / 1000000
sps_fnd_loc = sps_fnd_loc.drop(columns=['HeadQuarter', 'Amount($)'])
sps_fnd_loc = sps_fnd_loc.rename(columns={'Location':'City', 'Rating':'Avg. Rating','Salary':'Avg. Salary',
                                          'Company/Brand' :'Nr. Companies Funded','Amount($MM)': 'Sum Funding ($MM)'})

print(sps_fnd_loc)
print(sps_fnd_loc[sps_fnd_loc['Avg. Rating']>4])
#Only 1 company in that city

5. Create a scatterplot that shows the relationship between the `Avg. Salary` and the `Sum Funding ($MM)`. Which `City` stands out in terms of total funding received by companies and salary paid to their employees?

In [16]:
import seaborn as sns

sns.scatterplot(x='Avg. Salary', y='Sum Funding ($MM)', data=sps_fnd_loc)

NameError: name 'sps_fnd_loc' is not defined